# Tokenization Analysis

Evaluates BPE subword tokenization as an alternative to word-level vocab.

**Motivation:** Word-level vocab (4101 tokens, 55% hapax, 25-32% OOV) led to unstable training. See `analyze_corpus.ipynb` for the full corpus analysis.

**Requires:** `pip install sentencepiece`

## 1. Setup

In [ ]:
import csv
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sentencepiece as spm

ROOT = Path(".").resolve().parent.parent
SPLIT_DIR = ROOT / "data/usl-suspilne"

# Load and split data from per-split CSVs
splits = {"train": [], "dev": [], "test": []}
for split_name in splits:
    split_csv = SPLIT_DIR / f"{split_name}.csv"
    if not split_csv.exists():
        print(f"WARNING: {split_csv} not found")
        continue
    with open(split_csv, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f, delimiter="|"):
            splits[split_name].append(row["text_norm"])

# Word-level baseline stats
split_tokens = {}
for s in ["train", "dev", "test"]:
    split_tokens[s] = [tok for text in splits[s] for tok in text.split()]

train_counter = Counter(split_tokens["train"])
train_vocab = set(train_counter.keys())
vocab_size = len(train_counter)

print(f"Loaded {sum(len(v) for v in splits.values())} sentences")
print(f"Word-level vocab: {vocab_size} types")
for s in ["dev", "test"]:
    oov = sum(1 for t in split_tokens[s] if t not in train_vocab)
    print(f"  {s} OOV: {oov}/{len(split_tokens[s])} = {oov/len(split_tokens[s])*100:.1f}%")

## 2. BPE Tokenization Sweep

In [ ]:
VOCAB_SIZES = [500, 1000, 2000, 3000, 4000]
bpe_results = []

# Write training texts to temp file
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    for text in splits["train"]:
        f.write(text + "\n")
    train_file = f.name

for vs in VOCAB_SIZES:
    with tempfile.TemporaryDirectory() as tmpdir:
        prefix = str(Path(tmpdir) / "sp")
        spm.SentencePieceTrainer.train(
            input=train_file,
            model_prefix=prefix,
            vocab_size=vs,
            model_type="bpe",
            character_coverage=1.0,
            pad_id=3,
        )
        sp = spm.SentencePieceProcessor(model_file=prefix + ".model")

        result = {"vocab_size": vs}
        for s in ["train", "dev", "test"]:
            all_pieces = []
            lengths = []
            for text in splits[s]:
                pieces = sp.encode(text, out_type=str)
                all_pieces.extend(pieces)
                lengths.append(len(pieces))

            # Check for unknowns by ID
            unk_count = 0
            for text in splits[s]:
                ids = sp.encode(text, out_type=int)
                unk_count += sum(1 for i in ids if i == 0)

            result[f"{s}_avg_len"] = np.mean(lengths)
            result[f"{s}_total_tokens"] = len(all_pieces)
            result[f"{s}_oov"] = unk_count
            result[f"{s}_oov_pct"] = unk_count / len(all_pieces) * 100 if all_pieces else 0

        # Vocab utilization
        train_piece_types = set()
        for text in splits["train"]:
            train_piece_types.update(sp.encode(text, out_type=str))
        result["used_vocab"] = len(train_piece_types)
        result["vocab_util"] = len(train_piece_types) / vs * 100

        bpe_results.append(result)
        print(f"BPE {vs:>5}: train avg={result['train_avg_len']:.1f} tok/sent, "
              f"dev OOV={result['dev_oov']}, test OOV={result['test_oov']}, "
              f"vocab used={result['used_vocab']}/{vs} ({result['vocab_util']:.0f}%)")

Path(train_file).unlink()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

vs_list = [r["vocab_size"] for r in bpe_results]

# Fertility
for s, color in [("train", "blue"), ("dev", "orange"), ("test", "green")]:
    ax1.plot(vs_list, [r[f"{s}_avg_len"] for r in bpe_results], "o-", color=color, label=s)
    wl_avg = np.mean([len(t.split()) for t in splits[s]])
    ax1.axhline(wl_avg, color=color, linestyle=":", alpha=0.5)
ax1.set_xlabel("BPE vocab size")
ax1.set_ylabel("Avg tokens per sentence")
ax1.set_title("Fertility (dotted = word-level baseline)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Vocab utilization
ax2.plot(vs_list, [r["vocab_util"] for r in bpe_results], "ro-")
ax2.set_xlabel("BPE vocab size")
ax2.set_ylabel("Vocabulary utilization (%)")
ax2.set_title("% of vocab seen in training")
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

## 3. BPE vs Word-level Comparison

In [ ]:
dev_oov_tokens = [t for t in split_tokens["dev"] if t not in train_vocab]
test_oov_tokens = [t for t in split_tokens["test"] if t not in train_vocab]
wl_train_avg = np.mean([len(t.split()) for t in splits["train"]])

hapax = sum(1 for c in train_counter.values() if c == 1)
hapax_ratio = hapax / vocab_size * 100

print(f"{'Method':<16} {'Vocab':>6} {'Dev OOV%':>9} {'Test OOV%':>10} {'Train avg len':>14} {'Hapax %':>8}")
print("-" * 67)
print(f"{'Word-level':<16} {vocab_size:>6} {len(dev_oov_tokens)/len(split_tokens['dev'])*100:>8.1f}% "
      f"{len(test_oov_tokens)/len(split_tokens['test'])*100:>9.1f}% {wl_train_avg:>14.1f} {hapax_ratio:>7.1f}%")

for r in bpe_results:
    print(f"BPE {r['vocab_size']:<11} {r['vocab_size']:>6} {r['dev_oov_pct']:>8.1f}% "
          f"{r['test_oov_pct']:>9.1f}% {r['train_avg_len']:>14.1f} {'\u2014':>8}")

In [ ]:
# Example sentences: word-level vs BPE 2000
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    for text in splits["train"]:
        f.write(text + "\n")
    train_file = f.name

with tempfile.TemporaryDirectory() as tmpdir:
    prefix = str(Path(tmpdir) / "sp")
    spm.SentencePieceTrainer.train(
        input=train_file,
        model_prefix=prefix,
        vocab_size=2000,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=3,
    )
    sp2k = spm.SentencePieceProcessor(model_file=prefix + ".model")

    examples = []
    for text in splits["dev"]:
        toks = text.split()
        oov_pct = sum(1 for t in toks if t not in train_vocab) / max(len(toks), 1)
        examples.append((oov_pct, text))
    examples.sort(key=lambda x: x[0])
    indices = [0, len(examples)//4, len(examples)//2, 3*len(examples)//4, -1]

    print("Example tokenizations (dev set):\n")
    for idx in indices:
        oov_pct, text = examples[idx]
        bpe_pieces = sp2k.encode(text, out_type=str)
        word_toks = text.split()
        oov_words = [t for t in word_toks if t not in train_vocab]

        print(f"Text: {text[:100]}{'...' if len(text) > 100 else ''}")
        print(f"  Word-level: {len(word_toks)} tokens, OOV: {oov_words if oov_words else 'none'}")
        print(f"  BPE-2000:   {len(bpe_pieces)} tokens \u2192 {' '.join(bpe_pieces[:25])}{'...' if len(bpe_pieces) > 25 else ''}")
        print()

Path(train_file).unlink()

## 4. Embedding Parameter Budget

In [ ]:
EMBED_DIM = 256
TOTAL_PARAMS = 4_815_872  # from training log

wl_embed = (vocab_size + 4) * EMBED_DIM  # +4 special tokens
wl_total_tok = len(split_tokens["train"])

print(f"{'Method':<16} {'Vocab':>6} {'Embed params':>13} {'% of model':>11} {'Train tokens':>13} {'Tok/param':>10}")
print("-" * 73)
print(f"{'Word-level':<16} {vocab_size + 4:>6} {wl_embed:>13,} {wl_embed/TOTAL_PARAMS*100:>10.1f}% "
      f"{wl_total_tok:>13,} {wl_total_tok/wl_embed:>10.2f}")

for r in bpe_results:
    vs = r["vocab_size"]
    embed_params = vs * EMBED_DIM
    model_params = TOTAL_PARAMS - wl_embed + embed_params
    tok_total = r["train_total_tokens"]
    print(f"BPE {vs:<11} {vs:>6} {embed_params:>13,} {embed_params/model_params*100:>10.1f}% "
          f"{tok_total:>13,} {tok_total/embed_params:>10.2f}")

print()
print("Higher tok/param = more training signal per embedding parameter")

## 5. Recommendation

In [ ]:
bpe2k = next(r for r in bpe_results if r["vocab_size"] == 2000)

print("=" * 60)
print("RECOMMENDATION: BPE with vocab_size=2000")
print("=" * 60)
print()
print("Rationale:")
print(f"  1. OOV: {len(dev_oov_tokens)/len(split_tokens['dev'])*100:.0f}% (word) \u2192 {bpe2k['dev_oov_pct']:.0f}% (BPE) on dev")
print(f"  2. Vocab: {vocab_size} \u2192 {bpe2k['vocab_size']} tokens (51% smaller)")
print(f"  3. Embedding params: {wl_embed:,} \u2192 {bpe2k['vocab_size']*EMBED_DIM:,} (halved)")
print(f"  4. Fertility: ~{bpe2k['train_avg_len']:.0f} subwords/sent vs ~{wl_train_avg:.0f} words/sent")
print(f"  5. Every subword unit gets meaningful training signal")
print()
print("To apply:")
print("  python scripts/prepare_progressive_data.py --bpe 2000")